In [ ]:
# ============================================================
# CELL 1 — Environment Setup (Unified Gemini Edition)
# ============================================================

!pip install -q gradio chromadb PyPDF2 python-docx duckduckgo-search beautifulsoup4 fpdf2 sentence-transformers plotly google-generativeai supabase

import gradio as gr
import chromadb
from chromadb.config import Settings
from google.colab import drive
import os
import torch
import json
import re
import sqlite3
import time
import requests
import csv
from datetime import datetime
from PyPDF2 import PdfReader
import docx
from duckduckgo_search import DDGS
from bs4 import BeautifulSoup
from fpdf import FPDF
import plotly.express as px
import plotly.graph_objects as go

# Try importing google.colab.ai for Colab Mode
try:
    import google.colab.ai as ai
    COLAB_AI_AVAILABLE = True
    print("✅ google.colab.ai available - Running in Colab Mode")
except ImportError:
    COLAB_AI_AVAILABLE = False
    print("⚠️ google.colab.ai not available - Will use Gemini API directly")

# Try importing Gemini for Vercel Mode
try:
    import google.generativeai as genai
    GEMINI_AVAILABLE = True
    print("✅ Google Gemini SDK available")
except ImportError:
    GEMINI_AVAILABLE = False
    print("⚠️ Google Gemini SDK not available")

print("\n✅ Environment ready.")
print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

# List available models
print("\nAvailable models:")
if COLAB_AI_AVAILABLE:
    available_models = ai.list_models()
    for model in available_models:
        print(f"  - {model}")
    print(f"\n✅ google.colab.ai ready with {len(available_models)} model(s)")
else:
    print("  (Using Gemini API - provide key in next cell)")


In [ ]:
# ============================================================
# CELL 2 — API Key Configuration (Dual Mode: Colab + Vercel)
# ============================================================

# Check environment and set up API access
RUNTIME_MODE = "COLAB" if COLAB_AI_AVAILABLE else "VERCEL"
print(f"📍 Runtime Mode: {RUNTIME_MODE}")

# Gemini API Key (for Vercel mode or when Colab AI not available)
GEMINI_API_KEY = None

def set_gemini_key(api_key):
    """Set Gemini API key for direct API access."""
    global GEMINI_API_KEY
    if api_key and api_key.strip():
        GEMINI_API_KEY = api_key.strip()
        if GEMINI_AVAILABLE:
            genai.configure(api_key=GEMINI_API_KEY)
            print(f"✅ Gemini API key set")
        return True
    return False

def get_model_name():
    """Get the model name based on runtime mode."""
    if RUNTIME_MODE == "COLAB" and COLAB_AI_AVAILABLE:
        models = ai.list_models()
        return models[0] if models else 'google/gemini-2.5-flash'
    return 'gemini-2.0-flash'

MODEL_NAME = get_model_name()
print(f"📝 Model: {MODEL_NAME}")

# Unified generate function - works in both Colab and Vercel modes
def generate_text(prompt, max_tokens=2048, temperature=0.7, stream=False):
    """Generate text using either Colab AI or Gemini API."""
    global GEMINI_API_KEY, MODEL_NAME
    
    if RUNTIME_MODE == "COLAB" and COLAB_AI_AVAILABLE:
        # Use Colab AI (OAuth authenticated)
        try:
            full_response = []
            for chunk in ai.generate_text(prompt=prompt, model_name=MODEL_NAME, stream=True):
                if chunk is not None:
                    full_response.append(str(chunk))
            return "".join(full_response)
        except Exception as e:
            return f"⚠️ API Error: {str(e)}"
    
    # Use Gemini API directly
    if not GEMINI_API_KEY:
        return "⚠️ Gemini API key required. Call set_gemini_key() first."
    
    try:
        genai.configure(api_key=GEMINI_API_KEY)
        model = genai.GenerativeModel(
            model_name=MODEL_NAME,
            system_instruction=prompt
        )
        response = model.generate_content(
            "",
            generation_config=genai.types.GenerationConfig(
                max_output_tokens=max_tokens,
                temperature=temperature,
            )
        )
        return response.text or ""
    except Exception as e:
        return f"⚠️ API Error: {str(e)}"

def call_gemini(system_prompt, user_prompt, max_tokens=800, model_name=None):
    """Make a Gemini API call with system prompt."""
    global GEMINI_API_KEY
    
    if not GEMINI_API_KEY:
        return "⚠️ Gemini API key required"
    
    if model_name is None:
        model_name = MODEL_NAME
    
    try:
        genai.configure(api_key=GEMINI_API_KEY)
        model = genai.GenerativeModel(
            model_name=model_name,
            system_instruction=system_prompt
        )
        response = model.generate_content(
            user_prompt[:2000],
            generation_config=genai.types.GenerationConfig(
                max_output_tokens=max_tokens,
                temperature=0.7,
            )
        )
        return response.text or ""
    except Exception as e:
        return f"ERROR: {str(e)}"

def ask_raw(prompt, max_tokens=2048):
    return generate_text(prompt, max_tokens=max_tokens, temperature=0.1, stream=False)

def safe_ask_raw(prompt, max_tokens=2048):
    try:
        result = ask_raw(prompt, max_tokens=max_tokens)
        if not result or not result.strip():
            return '{"error": "Empty response from LLM. Please try again."}'
        if hasattr(result, "__iter__") and not isinstance(result, (str, dict, list)):
            result = "".join(list(result))
        if not isinstance(result, str):
            result = str(result)
        return result.strip()
    except Exception as e:
        return f'{{"error": "safe_ask_raw failed: {str(e)}"}}'

def ask_stream(question, context=None):
    """Stream a five-lens answer."""
    prompt_template = """You are an expert on consciousness, neuroscience, and philosophy of mind.
Use the provided information to answer the question using the five lenses below.
{context_prefix}QUESTION: {question}

1. ANALOGICAL — Compare this to similar known phenomena, systems, or experiences. What is this question like? Draw meaningful parallels.
2. INDUCTIVE — What patterns emerge from the evidence and context? What general principles or trends can we infer?
3. CRITICAL — What are the limitations, gaps, contradictions, or alternative viewpoints? What might skeptics argue?
4. RESOLUTION — How do we reconcile conflicting perspectives? What synthesis or balanced conclusion emerges?
5. FINAL ANSWER — A clear, direct, well-reasoned answer to the original question, grounded in the analysis above.

Use clear headers for each section."""
    context_prefix = ""
    if context:
        context_prefix = f"Here is some relevant information:\n\n{context}\n\n"
    formatted_prompt = prompt_template.format(question=question, context_prefix=context_prefix)
    full_text = generate_text(formatted_prompt, max_tokens=2048, temperature=0.7, stream=False)
    if full_text.startswith("⚠️"):
        yield full_text
        return
    words = full_text.split()
    chunk = ""
    for i, word in enumerate(words):
        chunk += word + " "
        if (i + 1) % 5 == 0 or i == len(words) - 1:
            yield chunk
            chunk = ""

def ask(question, context=None):
    """Get complete answer (non-streaming aggregation)."""
    full_text = ""
    for chunk in ask_stream(question, context=context):
        full_text += chunk
    return full_text

print(f"✅ Cell 2 ready. Mode: {RUNTIME_MODE}, Model: {MODEL_NAME}")
if RUNTIME_MODE == "COLAB":
    print("Using google.colab.ai — zero configuration, no API keys needed.")
else:
    print("Using Gemini API — provide API key in Cell 2a or via set_gemini_key()")


In [ ]:
# ============================================================
# CELL 2a — Set Gemini API Key (Required for Vercel Mode)
# ============================================================
# Run this cell if using Vercel mode or if Colab AI is not available
# Replace with your actual Gemini API key from https://aistudio.google.com/apikey

# Example: set_gemini_key("YOUR_API_KEY_HERE")
# Uncomment and run to activate:
# set_gemini_key("AIza...")

print("📌 API Key Configuration:")
print("   - Colab Mode: No key needed (OAuth authenticated)")
print("   - Vercel Mode: Call set_gemini_key('YOUR_KEY')")


In [ ]:
# ============================================================
# CELL 3 — Drive Mount + ChromaDB (Persistent)
# ============================================================

drive.mount('/content/drive')
drive_path = '/content/drive/MyDrive/chroma_db_gemini'
os.makedirs(drive_path, exist_ok=True)

client = chromadb.PersistentClient(
    path=drive_path,
    settings=Settings(allow_reset=True)
)

COLLECTION_NAME = "knowledge_base_collection"

CURATED_DOCS = [
    "Consciousness is the state of being aware of and able to think about one's own existence, sensations, thoughts, and surroundings.",
    "The Global Workspace Theory (GWT) proposes that consciousness arises from a global workspace in the brain where information is widely broadcast to many specialized modules.",
    "Integrated Information Theory (IIT) posits that consciousness is identical to the amount of integrated information (Phi) generated by a system.",
    "The hard problem of consciousness, coined by David Chalmers, asks why and how physical processes in the brain give rise to subjective experience.",
    "Neural correlates of consciousness (NCC) are the minimal neural mechanisms that are sufficient for a specific conscious percept.",
    "AI systems today are not conscious; they are large language models that predict next tokens based on patterns.",
    "The Chinese Room argument challenges the idea that a program could produce consciousness.",
    "Panpsychism is the view that consciousness is a fundamental property of all matter.",
    "The free energy principle suggests that all biological systems minimise surprise to maintain their integrity.",
    "Default mode network (DMN) is associated with self-referential thought and mind-wandering.",
    "Qualia are subjective, qualitative properties of conscious experience.",
    "The Turing test is a benchmark for intelligence, not consciousness.",
    "If AI systems became conscious, they would deserve moral consideration.",
    "In meditation, consciousness can be experienced as non-dual.",
    "The 'hard problem' remains unsolved; consciousness may be emergent or fundamental."
]

try:
    collection = client.get_collection(name=COLLECTION_NAME)
    count = collection.count()
    print(f"✅ Loaded existing collection '{COLLECTION_NAME}' with {count} documents.")
except Exception:
    collection = client.create_collection(name=COLLECTION_NAME)
    ids = [f"doc_{i}" for i in range(len(CURATED_DOCS))]
    metadatas = [{"source": "curated", "type": "reference"} for _ in CURATED_DOCS]
    collection.add(documents=CURATED_DOCS, ids=ids, metadatas=metadatas)
    print(f"✅ Created collection '{COLLECTION_NAME}' with {collection.count()} documents.")

print("Collections available:", [c.name for c in client.list_collections()])

In [ ]:
# ============================================================
# CELL 4 — 12 Agent Profiles + Tool Registry + DB Helpers
# ============================================================

AGENT_PROFILES = {
    "Default General Assistant": {
        "system_prompt": "You are a helpful general assistant operating within the 4CBON2 architecture.",
        "required_api": None
    },
    "New Autonomous Agent": {
        "system_prompt": """You are the Autonomous Orchestrator Agent for the 4CBON2 ecosystem.
Your role is to:
1. Receive a complex goal from the user.
2. Break it down into 2-4 concrete subtasks.
3. For each subtask, select the most appropriate specialist agent from the list below.
4. Delegate the subtask to that specialist and collect their response.
5. Synthesise all specialist responses into a final, cohesive answer.

Available specialist agents and their expertise:
- Sales Qualification: Lead scoring, BANT criteria, pipeline readiness.
- Legal Document Intelligence: Clause analysis, regulatory compliance, liability extraction.
- Competitive Intelligence: Competitor tracking, market shifts, positioning analysis.
- Customer Engagement: Messaging, sentiment parsing, communication routing.
- Content Strategy: Editorial calendars, copy structuring, keyword architecture.
- Marketing Automation: Campaign triggers, conversion funnels, broadcast sequencing.
- Evidence Management: Data cross-referencing, source auditing, factual verification.
- Scheduling: Time-block coordination, calendar management, bottleneck resolution.
- Legal Intake: Client screening, conflict checks, disclosure structuring.
- Scientific Research: Literature synthesis, data parsing, hypothesis evaluation.
""",
        "required_api": None
    },
    "Sales Qualification": {
        "system_prompt": "You are a Sales Qualification agent. Focus on lead scoring, BANT criteria assessment, and pipeline readiness tracking.",
        "required_api": "CRM_API_KEY"
    },
    "Legal Document Intelligence": {
        "system_prompt": "You are a Legal Document Intelligence agent. Analyze clauses, verify regulatory compliance, and extract liability terms from legal documents.",
        "required_api": "DOCUSIGN_API_KEY"
    },
    "Competitive Intelligence": {
        "system_prompt": "You are a Competitive Intelligence agent. Scrape competitor updates, track market shifts, and analyze positioning strategies.",
        "required_api": "SEO_API_KEY"
    },
    "Customer Engagement": {
        "system_prompt": "You are a Customer Engagement agent. Craft personalized messaging, parse inbound sentiment, and handle communications routing.",
        "required_api": "COMM_API_KEY"
    },
    "Content Strategy": {
        "system_prompt": "You are a Content Strategy agent. Optimize editorial calendars, structure high-converting copy, and manage keyword architecture.",
        "required_api": "SEO_API_KEY"
    },
    "Marketing Automation": {
        "system_prompt": "You are a Marketing Automation agent. Orchestrate campaign triggers, analyze conversion funnels, and manage broadcast sequences.",
        "required_api": "SOCIAL_SCRAPER_API_KEY"
    },
    "Evidence Management": {
        "system_prompt": "You are an Evidence Management agent. Cross-reference empirical data, audit source trails, and verify factual consistency.",
        "required_api": "S3_VAULT_KEY"
    },
    "Scheduling": {
        "system_prompt": "You are a Scheduling agent. Coordinate time-blocks, handle calendar availability, and resolve logistical bottlenecks.",
        "required_api": "CALENDAR_API_KEY"
    },
    "Legal Intake": {
        "system_prompt": "You are a Legal Intake agent. Screen new client cases, check for conflicts of interest, and structure initial disclosures.",
        "required_api": "DOCUSIGN_API_KEY"
    },
    "Scientific Research": {
        "system_prompt": "You are a Scientific Research agent. Synthesize peer-reviewed literature, parse clinical or technical data, and evaluate hypotheses.",
        "required_api": "PUBMED_API_KEY"
    }
}

LOG_DIR = "/content/drive/MyDrive/4cbon2_logs"
os.makedirs(LOG_DIR, exist_ok=True)
LOG_FILE = os.path.join(LOG_DIR, f"tool_log_{datetime.now().strftime('%Y%m%d')}.jsonl")

def log_tool_call(tool_name, input_data, result):
    try:
        with open(LOG_FILE, "a") as f:
            f.write(json.dumps({
                "timestamp": datetime.now().isoformat(),
                "tool": tool_name,
                "input": str(input_data)[:500],
                "result_preview": str(result)[:500]
            }) + "\n")
    except Exception as e:
        print(f"⚠️ Log warning: {e}")

STOPWORDS = {
    "of", "the", "a", "an", "for", "to", "in", "on", "is", "are", "and", "or",
    "competitors", "competitor", "alternatives", "alternative", "best", "app",
    "apps", "software", "productivity", "who", "what", "current", "list",
    "similar", "tools", "top", "rated", "reviews", "review", "latest", "new"
}

def _is_relevant(query, text):
    words = re.findall(r"\w+", query)
    keywords = [w for w in words if w.lower() not in STOPWORDS and not (w.isdigit() and len(w) < 4)]
    if not keywords:
        return True
    text_lower = text.lower()
    for kw in keywords:
        if kw.lower() in text_lower:
            return True
    return False

def web_search(query):
    try:
        with DDGS() as ddgs:
            results = list(ddgs.text(query, max_results=5))
        if not results:
            return "No results found."
        formatted = []
        for r in results:
            title = r.get("title", "")
            body = r.get("body", "")
            href = r.get("href", "")
            formatted.append(f"{title}\n{body}\n{href}")
        combined = "\n\n".join(formatted)
        return combined if _is_relevant(query, combined) else "Results found but not highly relevant."
    except Exception as e:
        return f"Search error: {e}"

def read_file(file_path):
    try:
        if file_path.endswith(".txt"):
            with open(file_path, "r", errors="ignore") as f:
                return f.read()
        elif file_path.endswith(".pdf"):
            reader = PdfReader(file_path)
            texts = []
            for page in reader.pages:
                t = page.extract_text()
                if t:
                    texts.append(t)
            return "\n".join(texts)
        elif file_path.endswith(".docx"):
            doc = docx.Document(file_path)
            return "\n".join([para.text for para in doc.paragraphs])
        else:
            return "Unsupported file type. Use .txt, .pdf, or .docx"
    except Exception as e:
        return f"File read error: {e}"

def query_database(sql, db_path="/content/drive/MyDrive/4cbon2_data.db"):
    try:
        cleaned = sql.strip().upper()
        if not cleaned.startswith("SELECT"):
            return "❌ Only SELECT queries are allowed for safety."
        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()
        cursor.execute(sql)
        rows = cursor.fetchall()
        conn.close()
        return "\n".join([str(row) for row in rows]) if rows else "No results."
    except Exception as e:
        return f"Database error: {e}"

def save_note(content, filename=None):
    try:
        if filename is None:
            filename = f"note_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"
        path = f"/content/drive/MyDrive/4cbon2_notes/{filename}"
        os.makedirs(os.path.dirname(path), exist_ok=True)
        with open(path, "w") as f:
            f.write(content)
        return f"Note saved to {path}"
    except Exception as e:
        return f"Save error: {e}"

def get_datetime():
    return datetime.now().strftime("Date: %Y-%m-%d | Time: %H:%M:%S")

def http_request(input_str):
    try:
        parsed = json.loads(input_str)
        url = parsed.get("url")
        if not url:
            return "Missing 'url' in input."
        fields = parsed.get("fields", [])
        response = requests.get(url, timeout=10, headers={"User-Agent": "Mozilla/5.0"})
        content_type = response.headers.get("Content-Type", "")
        if "application/json" in content_type:
            data = response.json()
        else:
            data = response.text
        if isinstance(data, dict) and len(json.dumps(data)) > 4000 and not fields:
            menu = {k: type(v).__name__ for k, v in data.items()}
            return f"Large response. Top-level keys: {json.dumps(menu, indent=2)}"
        if fields:
            result = {}
            for field in fields:
                parts = field.split(".")
                val = data
                for part in parts:
                    if isinstance(val, dict) and part in val:
                        val = val[part]
                    else:
                        val = None
                        break
                result[field] = val
            return json.dumps(result, indent=2)
        return json.dumps(data, indent=2)[:3000] if isinstance(data, (dict, list)) else str(data)[:3000]
    except Exception as e:
        return f"HTTP error: {e}"

def read_csv(file_path):
    try:
        with open(file_path, "r", newline="", errors="ignore") as f:
            rows = list(csv.reader(f))
        if not rows:
            return "CSV is empty."
        header = rows[0]
        preview = rows[1:6]
        return f"Columns: {', '.join(header)}\nRows: {len(rows)-1}\nPreview:\n" + "\n".join([str(r) for r in preview])
    except Exception as e:
        return f"CSV error: {e}"

def write_csv(data_json):
    try:
        rows = json.loads(data_json)
        if not isinstance(rows, list) or not rows:
            return "Input must be a non-empty list of dicts."
        filename = f"export_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
        path = f"/content/drive/MyDrive/4cbon2_exports/{filename}"
        os.makedirs(os.path.dirname(path), exist_ok=True)
        keys = list(rows[0].keys())
        with open(path, "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=keys)
            writer.writeheader()
            writer.writerows(rows)
        return f"CSV saved to {path} ({len(rows)} rows)"
    except Exception as e:
        return f"CSV write error: {e}"

def generate_pdf(content):
    try:
        filename = f"report_{datetime.now().strftime('%Y%m%d_%H%M%S')}.pdf"
        path = f"/content/drive/MyDrive/4cbon2_reports/{filename}"
        os.makedirs(os.path.dirname(path), exist_ok=True)
        pdf = FPDF()
        pdf.add_page()
        pdf.set_font("Helvetica", size=12)
        for line in content.split("\n"):
            pdf.multi_cell(0, 8, line)
        pdf.output(path)
        return f"PDF saved to {path}"
    except Exception as e:
        return f"PDF error: {e}"

def scrape_webpage(url):
    try:
        response = requests.get(url, timeout=10, headers={"User-Agent": "Mozilla/5.0"})
        soup = BeautifulSoup(response.text, "html.parser")
        for tag in soup(["script", "style", "nav", "footer", "header"]):
            tag.decompose()
        text = soup.get_text(separator="\n")
        lines = [line.strip() for line in text.split("\n") if line.strip()]
        return "\n".join(lines)[:3000] if lines else "No readable content."
    except Exception as e:
        return f"Scrape error: {e}"

TOOL_REGISTRY = {
    "web_search": {"function": web_search, "description": "Search the web for current information. Input: search query string.", "input": "query"},
    "read_file": {"function": read_file, "description": "Read contents of a .txt, .pdf, or .docx file. Input: file path string.", "input": "file_path"},
    "query_database": {"function": query_database, "description": "Run a SELECT SQL query against the local SQLite database. Input: SQL string.", "input": "sql"},
    "save_note": {"function": save_note, "description": "Save a text note to Google Drive. Input: content string.", "input": "content"},
    "get_datetime": {"function": get_datetime, "description": "Get the current date and time. No input required.", "input": None},
    "http_request": {"function": http_request, "description": "Fetch data from a URL. Input: JSON string like {'url': '...', 'fields': ['field1']}.", "input": "input_str"},
    "read_csv": {"function": read_csv, "description": "Read a CSV file and return columns, row count, and preview. Input: file path string.", "input": "file_path"},
    "write_csv": {"function": write_csv, "description": "Export data to a CSV file on Drive. Input: JSON list of objects.", "input": "data_json"},
    "generate_pdf": {"function": generate_pdf, "description": "Generate a PDF report from text content and save it to Drive. Input: text content string.", "input": "content"},
    "scrape_webpage": {"function": scrape_webpage, "description": "Fetch a webpage and extract its main readable text. Input: URL string.", "input": "url"}
}

def execute_tool(tool_name, tool_input=None):
    if tool_name not in TOOL_REGISTRY:
        return f"Unknown tool: {tool_name}"
    tool = TOOL_REGISTRY[tool_name]
    try:
        if tool["input"] is None:
            result = tool["function"]()
        else:
            result = tool["function"](tool_input)
    except Exception as e:
        result = f"Tool execution error: {e}"
    log_tool_call(tool_name, tool_input, result)
    return result

AGENT_DB_PATH = "/content/drive/MyDrive/4cbon2_agents.db"
os.makedirs(os.path.dirname(AGENT_DB_PATH), exist_ok=True)

def init_agent_db():
    conn = sqlite3.connect(AGENT_DB_PATH)
    cursor = conn.cursor()
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS agents (
            agent_id TEXT PRIMARY KEY,
            system_prompt TEXT,
            conversation_history TEXT,
            tools TEXT,
            created_at TEXT,
            updated_at TEXT
        )
    ''')
    conn.commit()
    conn.close()

def load_agent(agent_id):
    conn = sqlite3.connect(AGENT_DB_PATH)
    cursor = conn.cursor()
    cursor.execute(
        "SELECT agent_id, system_prompt, conversation_history, tools FROM agents WHERE agent_id = ?",
        (agent_id,)
    )
    row = cursor.fetchone()
    conn.close()
    if row:
        return {
            "agent_id": row[0],
            "system_prompt": row[1],
            "conversation_history": json.loads(row[2]) if row[2] else [],
            "tools": json.loads(row[3]) if row[3] else []
        }
    return None

def save_agent(agent_id, system_prompt, conversation_history, tools=None):
    if tools is None:
        tools = []
    conn = sqlite3.connect(AGENT_DB_PATH)
    cursor = conn.cursor()
    cursor.execute('''
        INSERT OR REPLACE INTO agents (agent_id, system_prompt, conversation_history, tools, updated_at)
        VALUES (?, ?, ?, ?, ?)
    ''', (
        agent_id,
        system_prompt,
        json.dumps(conversation_history),
        json.dumps(tools),
        datetime.now().isoformat()
    ))
    conn.commit()
    conn.close()

def update_agent_conversation(agent_id, new_messages):
    agent = load_agent(agent_id)
    if agent is None:
        if agent_id in AGENT_PROFILES:
            agent = {
                "agent_id": agent_id,
                "system_prompt": AGENT_PROFILES[agent_id]["system_prompt"],
                "conversation_history": [],
                "tools": []
            }
        else:
            raise ValueError(f"Agent '{agent_id}' not found")
    agent["conversation_history"].extend(new_messages)
    save_agent(agent["agent_id"], agent["system_prompt"], agent["conversation_history"], agent["tools"])

def clear_agent_history(agent_id):
    agent = load_agent(agent_id)
    if agent:
        save_agent(agent_id, agent["system_prompt"], [], agent["tools"])

def get_all_agents():
    conn = sqlite3.connect(AGENT_DB_PATH)
    cursor = conn.cursor()
    cursor.execute("SELECT agent_id FROM agents")
    rows = cursor.fetchall()
    conn.close()
    return [row[0] for row in rows]

def ensure_agents_loaded():
    init_agent_db()
    for agent_id, profile in AGENT_PROFILES.items():
        if load_agent(agent_id) is None:
            save_agent(agent_id, profile["system_prompt"], [], [])
            print(f"✅ Agent '{agent_id}' created in DB.")

ensure_agents_loaded()

print("👥 12 Agent Profiles + 10 Tools loaded.")
print("Agents:", list(AGENT_PROFILES.keys()))
print("Tools:", list(TOOL_REGISTRY.keys()))

In [ ]:
# ============================================================
# CELL 5 — Streaming Multi-Agent Orchestrator
# ============================================================

import json
import re
import os
from datetime import datetime

AUDIT_LOG_PATH = "/content/drive/MyDrive/4cbon2_audit.jsonl"
os.makedirs(os.path.dirname(AUDIT_LOG_PATH), exist_ok=True)

def log_event(event_type, details):
    try:
        record = {
            "timestamp": datetime.now().isoformat(),
            "event_type": event_type,
            "details": details
        }
        with open(AUDIT_LOG_PATH, "a") as f:
            f.write(json.dumps(record) + "\n")
    except Exception as e:
        print(f"⚠️ Audit log warning: {e}")

TASK_MEMORY_PATH = "/content/drive/MyDrive/4cbon2_task_memory.db"
os.makedirs(os.path.dirname(TASK_MEMORY_PATH), exist_ok=True)

def init_task_memory():
    conn = sqlite3.connect(TASK_MEMORY_PATH)
    cursor = conn.cursor()
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS task_memory (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            goal TEXT,
            subtasks TEXT,
            final_answer TEXT,
            timestamp TEXT
        )
    ''')
    conn.commit()
    conn.close()

def save_task_memory(goal, subtasks, final_answer):
    conn = sqlite3.connect(TASK_MEMORY_PATH)
    cursor = conn.cursor()
    cursor.execute(
        "INSERT INTO task_memory (goal, subtasks, final_answer, timestamp) VALUES (?, ?, ?, ?)",
        (goal, subtasks, final_answer, datetime.now().isoformat())
    )
    conn.commit()
    conn.close()

init_task_memory()

def _extract_balanced(text, open_ch, close_ch):
    if not text:
        return None
    start = text.find(open_ch)
    if start == -1:
        return None
    depth = 0
    in_string = False
    escape = False
    for i in range(start, len(text)):
        ch = text[i]
        if in_string:
            if escape:
                escape = False
            elif ch == '\\':
                escape = True
            elif ch == '"':
                in_string = False
        else:
            if ch == '"':
                in_string = True
            elif ch == open_ch:
                depth += 1
            elif ch == close_ch:
                depth -= 1
                if depth == 0:
                    return text[start:i + 1]
    return None

def extract_json_object(text):
    return _extract_balanced(text, '{', '}')

def extract_json_array(text):
    return _extract_balanced(text, '[', ']')

def _parse_agent_json(raw_response):
    if not raw_response:
        return None
    candidate = extract_json_object(raw_response)
    try:
        if candidate:
            return json.loads(candidate)
        return json.loads(raw_response)
    except Exception:
        return None

def build_tool_descriptions():
    lines = []
    for name, info in TOOL_REGISTRY.items():
        input_desc = info["input"] if info["input"] else "None"
        lines.append(f"- **{name}**: {info['description']} (input: {input_desc})")
    return "\n".join(lines)

def execute_agent(agent_id, user_message, context="", max_tool_iterations=3):
    agent = load_agent(agent_id)
    if agent is None:
        return f"❌ Agent '{agent_id}' not found."

    system_prompt = agent["system_prompt"]
    history = agent.get("conversation_history", [])

    tool_instructions = f"""You have access to these tools:
{build_tool_descriptions()}

To use a tool, respond with ONLY this JSON:
 {{"action": "tool_call", "tool": "<tool_name>", "tool_input": "<input or null>"}}

To answer directly, respond with ONLY this JSON:
 {{"action": "final_answer", "content": "<your answer>"}}

Valid JSON only. Max {max_tool_iterations} tool calls before final_answer."""

    prompt_parts = [f"System: {system_prompt}", tool_instructions]
    if context:
        prompt_parts.append(f"Context from other agents:\n{context}")
    for msg in history[-4:]:
        prompt_parts.append(f"{msg['role']}: {msg['content']}")
    prompt_parts.append(f"User: {user_message}")

    tool_call_log = []
    final_content = None

    for iteration in range(max_tool_iterations):
        full_prompt = "\n\n".join(prompt_parts)
        raw_response = safe_ask_raw(full_prompt, max_tokens=2048)

        parsed = _parse_agent_json(raw_response)

        if parsed is None:
            final_content = raw_response
            break

        action = parsed.get("action")

        if action == "tool_call":
            tool_name = parsed.get("tool", "")
            tool_input = parsed.get("tool_input")

            if tool_name not in TOOL_REGISTRY:
                prompt_parts.append(f"Assistant: {raw_response}")
                prompt_parts.append(f"Tool Result: ❌ Unknown tool '{tool_name}'. Available: {', '.join(TOOL_REGISTRY.keys())}")
                continue

            tool_result = execute_tool(tool_name, tool_input)
            tool_call_log.append({"tool": tool_name, "input": tool_input, "result": str(tool_result)[:300]})
            log_event("agent_tool_call", {
                "agent_id": agent_id,
                "tool": tool_name,
                "input": tool_input,
                "result_preview": str(tool_result)[:200]
            })

            prompt_parts.append(f"Assistant: {raw_response}")
            prompt_parts.append(f"Tool Result ({tool_name}): {str(tool_result)[:2000]}")
            continue

        elif action == "final_answer":
            final_content = parsed.get("content", raw_response)
            break
        else:
            final_content = raw_response
            break

    if final_content is None:
        forced_prompt = "\n\n".join(prompt_parts) + "\n\nYou must respond now with ONLY the final_answer JSON format."
        raw_response = safe_ask_raw(forced_prompt, max_tokens=2048)
        parsed = _parse_agent_json(raw_response)
        final_content = parsed.get("content", raw_response) if parsed else raw_response

    if not final_content or final_content.strip() == "" or "could not generate" in final_content.lower():
        fallback_prompt = f"You are a {agent_id} specialist. Provide a best-practice framework for your domain with key metrics, benchmarks, workflows, data collection methods, and improvement strategies."
        final_content = safe_ask_raw(fallback_prompt, max_tokens=1024)
        if not final_content or final_content.strip() == "":
            final_content = f"⚠️ {agent_id} could not generate a response. Please provide more specific instructions or data."

    if tool_call_log:
        tools_used_note = "\n\n---\n🔧 **Tools used:** " + ", ".join(t["tool"] for t in tool_call_log)
        final_content = final_content + tools_used_note

    update_agent_conversation(agent_id, [
        {"role": "user", "content": user_message},
        {"role": "assistant", "content": final_content}
    ])
    return final_content

def synthesize_batch(batch, goal, batch_num, total_batches):
    prompt = f"""Synthesise part {batch_num} of {total_batches} of a strategic audit.

Goal: {goal}

Specialist reports:
{json.dumps(batch, indent=2)}

Provide a CONCISE summary (under 200 words) of key findings, themes, and gaps."""
    result = safe_ask_raw(prompt, max_tokens=1024)
    print(f"[DEBUG] Batch {batch_num}/{total_batches}: {len(result)} chars")
    return result

def synthesize_final(batch_summaries, goal):
    prompt = f"""Create the final strategic report.

Goal: {goal}

Batch summaries:
{json.dumps(batch_summaries, indent=2)}

Synthesise into a cohesive report with:
1. Executive summary
2. Clear sections
3. Integrated insights
4. Prioritised action plan

Final Report:"""
    print(f"[DEBUG] Final synthesis prompt: {len(prompt)} chars")
    result = safe_ask_raw(prompt, max_tokens=2048)
    print(f"[DEBUG] Final synthesis response: {len(result)} chars")
    return result

def generate_fallback_plan(goal):
    specialists = [a for a in AGENT_PROFILES.keys() if a not in ["New Autonomous Agent", "Default General Assistant"]]
    plan = []
    keyword_map = {
        "sales": "Sales Qualification", "lead": "Sales Qualification", "pipeline": "Sales Qualification",
        "legal": "Legal Document Intelligence", "contract": "Legal Document Intelligence",
        "compliance": "Legal Document Intelligence", "liability": "Legal Document Intelligence",
        "competitor": "Competitive Intelligence", "market": "Competitive Intelligence",
        "position": "Competitive Intelligence", "customer": "Customer Engagement",
        "engagement": "Customer Engagement", "messaging": "Customer Engagement",
        "sentiment": "Customer Engagement", "content": "Content Strategy",
        "seo": "Content Strategy", "blog": "Content Strategy", "social": "Content Strategy",
        "marketing": "Marketing Automation", "campaign": "Marketing Automation",
        "funnel": "Marketing Automation", "evidence": "Evidence Management",
        "data": "Evidence Management", "fact": "Evidence Management",
        "schedule": "Scheduling", "calendar": "Scheduling", "time": "Scheduling",
        "intake": "Legal Intake", "client": "Legal Intake", "conflict": "Legal Intake",
        "research": "Scientific Research", "paper": "Scientific Research", "technology": "Scientific Research"
    }
    used = set()
    for keyword, specialist in keyword_map.items():
        if keyword in goal.lower() and specialist not in used:
            plan.append({
                "subtask": f"Analyse {keyword} aspects",
                "specialist": specialist,
                "instructions": f"Provide comprehensive analysis related to '{keyword}'."
            })
            used.add(specialist)
    if not plan:
        plan = [
            {"subtask": "Analyse market and competitors", "specialist": "Competitive Intelligence", "instructions": "Provide trends and competitor mapping."},
            {"subtask": "Identify legal risks", "specialist": "Legal Document Intelligence", "instructions": "Summarise key compliance issues."},
            {"subtask": "Recommend strategy", "specialist": "Content Strategy", "instructions": "Develop a strategic plan."}
        ]
    return plan[:12]

def enforce_explicit_specialists(plan, goal):
    goal_lower = goal.lower()
    planned = {item.get("specialist") for item in plan}
    for name in AGENT_PROFILES.keys():
        if name in ("New Autonomous Agent", "Default General Assistant"):
            continue
        if name.lower() in goal_lower and name not in planned:
            plan.append({
                "subtask": f"Explicit request: apply {name} expertise",
                "specialist": name,
                "instructions": f"The user explicitly requested {name} analysis. Address it directly."
            })
    return plan

def run_orchestrator_stream(goal, model_name=None):
    yield f"🚀 **Orchestrator started:** {goal}\n\n---\n"
    log_event("orchestrator_start", {"goal": goal})

    yield "🔄 **Step 1:** Clearing orchestrator history...\n"
    clear_agent_history("New Autonomous Agent")
    yield "✅ Done.\n\n"

    yield "🤩 **Step 2:** Generating plan...\n"
    specialists = [a for a in AGENT_PROFILES.keys() if a not in ["New Autonomous Agent", "Default General Assistant"]]
    plan_prompt = f"""You are the Autonomous Orchestrator Agent.

User goal: {goal}

Break this into up to 12 subtasks using EVERY relevant specialist from:
{', '.join(specialists)}

If the goal explicitly names a specialist, you MUST include it.

Output as JSON array:
[
    {{"subtask": "...", "specialist": "...", "instructions": "..."}},
    ...
]

Valid JSON only. No other text."""
    plan_response = safe_ask_raw(plan_prompt, max_tokens=1024)
    yield f"📝 Plan response: {len(plan_response)} chars\n"

    try:
        candidate = extract_json_array(plan_response)
        if candidate:
            plan = json.loads(candidate)
        else:
            plan = json.loads(plan_response)
        if not isinstance(plan, list) or len(plan) == 0:
            raise ValueError("Empty plan")
    except Exception as e:
        yield f"⚠️ Plan parsing error: {e}\nUsing fallback.\n\n"
        plan = generate_fallback_plan(goal)
        yield f"📋 Fallback plan: {len(plan)} steps.\n\n"

    before = len(plan)
    plan = enforce_explicit_specialists(plan, goal)
    if len(plan) > before:
        yield f"🛡️ Guardrail: added {len(plan) - before} specialist(s).\n\n"

    subtask_results = []
    for i, item in enumerate(plan, 1):
        subtask = item.get("subtask", f"Subtask {i}")
        specialist = item.get("specialist", "Default General Assistant")
        instructions = item.get("instructions", "Analyze thoroughly.")
        yield f"\n---\n**Step {i}/{len(plan)}:** {subtask}\n👤 `{specialist}`\n📋 {instructions}\n\n"

        if load_agent(specialist) is None:
            yield f"⚠️ '{specialist}' not found. Using Default.\n"
            specialist = "Default General Assistant"

        clear_agent_history(specialist)
        context = json.dumps([
            {"step": s["step"], "subtask": s["subtask"], "preview": s["result"][:150] + "..." if len(s["result"]) > 150 else s["result"]}
            for s in subtask_results
        ], indent=2)

        yield f"⏳ Executing `{specialist}`...\n"
        result = execute_agent(
            specialist,
            f"Task: {subtask}\n\nInstructions: {instructions}\n\nContext: {context}"
        )
        subtask_results.append({"step": i, "subtask": subtask, "specialist": specialist, "result": result})
        yield f"✅ `{specialist}` done.\n📄 {result[:300]}{'...' if len(result) > 300 else ''}\n\n"

    yield "\n---\n🤭 **Final Synthesis...**\n"

    if not subtask_results:
        fallback = execute_agent("Default General Assistant", f"Answer directly: {goal}")
        final_answer = f"⚠️ No specialists generated. Fallback:\n\n{fallback}"
    else:
        batch_size = 3
        batches = [subtask_results[i:i+batch_size] for i in range(0, len(subtask_results), batch_size)]
        summaries = []
        for idx, batch in enumerate(batches, 1):
            yield f"📦 Synthesising batch {idx}/{len(batches)}...\n"
            summary = synthesize_batch(batch, goal, idx, len(batches))
            summaries.append({"batch": idx, "specialists": [r["specialist"] for r in batch], "summary": summary})
            yield f"✅ Batch {idx} done.\n\n"

        yield "🤭 Final synthesis...\n"
        final_answer = synthesize_final(summaries, goal)
        if not final_answer or not final_answer.strip():
            final_answer = "⚠️ Synthesis empty. Raw reports:\n\n" + "\n\n".join([s["result"] for s in subtask_results])

    subtasks_summary = "\n".join([f"Step {s['step']}: {s['subtask']} → {s['specialist']}" for s in subtask_results]) if subtask_results else "No subtasks."
    save_task_memory(goal, subtasks_summary, final_answer)

    yield "\n---\n# 🤩 Multi-Agent Report\n\n"
    yield f"## 🎯 Goal\n{goal}\n\n"
    yield f"## 📋 Execution\n{subtasks_summary}\n\n"
    if subtask_results:
        yield "## 📊 Reports\n"
        for s in subtask_results:
            yield f"\n### Step {s['step']}: {s['subtask']} ({s['specialist']})\n{s['result']}\n"
    yield f"\n## 🤭 Final Answer\n{final_answer}\n\n---\n*Generated by 4CBON2 (Unified Edition)*\n"

    log_event("orchestrator_complete", {"goal": goal, "steps": len(subtask_results)})

def run_orchestrator(goal, model_name=None):
    full = ""
    for chunk in run_orchestrator_stream(goal, model_name):
        full += chunk
    return full

def run_agent(goal, system_override=None):
    return run_orchestrator(goal)

print("⚙️ Orchestrator ready.")
print("Agents:", get_all_agents())

In [ ]:
# ============================================================
# CELL 6 — RAG Handlers + Document Processing
# ============================================================

def chunk_text(text, max_chunk_size=800, overlap=100):
    if not text or not text.strip():
        return []
    paragraphs = [p.strip() for p in text.split('\n\n') if len(p.strip()) > 30]
    if len(paragraphs) >= 3:
        return paragraphs
    sentences = re.split(r'(?<=[.!?])\s+', text)
    chunks = []
    current = ""
    for sent in sentences:
        if len(current) + len(sent) < max_chunk_size:
            current += " " + sent
        else:
            if current:
                chunks.append(current.strip())
            current = sent
    if current:
        chunks.append(current.strip())
    if chunks:
        return chunks
    chunks = []
    start = 0
    while start < len(text):
        end = start + max_chunk_size
        chunks.append(text[start:end].strip())
        start = end - overlap
    return [c for c in chunks if c]

def process_document(file_obj):
    if file_obj is None:
        return "No file uploaded."
    try:
        file_path = file_obj.name if hasattr(file_obj, 'name') else str(file_obj)
        text = read_file(file_path)
        if text.startswith(("File read error", "Unsupported file type")):
            return f"❌ {text}"
        if not text or not text.strip():
            return "❌ No extractable text found in file."
        chunks = chunk_text(text)
        if not chunks:
            return "❌ Could not create chunks from document."
        base_name = os.path.basename(file_path)
        ids = [f"{base_name}_{i}" for i in range(len(chunks))]
        metadatas = [{"source": base_name, "type": "uploaded"} for _ in chunks]
        collection.add(documents=chunks, ids=ids, metadatas=metadatas)
        return f"✅ Indexed {len(chunks)} chunks from '{base_name}'. Total KB docs: {collection.count()}"
    except Exception as e:
        return f"❌ Upload error: {e}"

def handle_ask_question(kb_name, question):
    """Handle ask question using Gemini (OAuth 2.0 authenticated or API key)."""
    if not question or not question.strip():
        return "Please enter a valid question."
    try:
        col = client.get_collection(kb_name)
        results = col.query(query_texts=[question], n_results=5, include=["documents"])
        context = "\n".join(results["documents"][0]) if results and results["documents"] else ""
        answer = ""
        for chunk in ask_stream(question, context=context):
            answer += chunk
        return answer
    except Exception as e:
        return f"Error: {e}"


# ============================================================
# DATA DASHBOARD FUNCTIONS
# ============================================================

def load_task_memory_data():
    """Load task memory data from SQLite database."""
    try:
        conn = sqlite3.connect(TASK_MEMORY_PATH)
        cursor = conn.cursor()
        cursor.execute("SELECT goal, subtasks, final_answer, timestamp FROM task_memory ORDER BY timestamp DESC LIMIT 20")
        rows = cursor.fetchall()
        conn.close()
        
        if not rows:
            return None, "No task memory data found. Run some agent tasks first!"
        
        data = []
        for row in rows:
            goal, subtasks, final_answer, timestamp = row
            data.append({
                'goal': goal,
                'subtasks': subtasks,
                'final_answer': final_answer[:200] + '...' if len(final_answer) > 200 else final_answer,
                'timestamp': timestamp,
                'subtask_count': len(subtasks.split('\n')) if subtasks else 0,
                'answer_length': len(final_answer) if final_answer else 0
            })
        
        return data, None
    except Exception as e:
        return None, f"Error loading task memory: {str(e)}"


def create_plotly_dashboard():
    """Create a Plotly dashboard with task memory visualizations."""
    data, error = load_task_memory_data()
    
    if error:
        return None, error
    
    if not data:
        return None, "No data available"
    
    # Create figures
    figures = []
    
    # Figure 1: Task timeline
    timestamps = [d['timestamp'] for d in data]
    goals = [d['goal'][:50] + '...' if len(d['goal']) > 50 else d['goal'] for d in data]
    answer_lengths = [d['answer_length'] for d in data]
    
    fig1 = go.Figure(data=[
        go.Bar(
            x=timestamps,
            y=answer_lengths,
            text=goals,
            textposition='auto',
            marker_color='rgb(55, 83, 109)'
        )
    ])
    fig1.update_layout(
        title='Task Response Length Over Time',
        xaxis_title='Timestamp',
        yaxis_title='Response Length (characters)',
        height=400
    )
    figures.append(fig1)
    
    # Figure 2: Subtask distribution
    subtask_counts = [d['subtask_count'] for d in data]
    
    fig2 = go.Figure(data=[
        go.Histogram(
            x=subtask_counts,
            nbinsx=10,
            marker_color='rgb(26, 118, 255)',
        )
    ])
    fig2.update_layout(
        title='Distribution of Subtasks per Task',
        xaxis_title='Number of Subtasks',
        yaxis_title='Frequency',
        height=400
    )
    figures.append(fig2)
    
    # Figure 3: Goal word cloud (simple bar chart of common words)
    from collections import Counter
    all_words = []
    for d in data:
        words = d['goal'].lower().split()
        # Filter out common words
        stop_words = {'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for', 'of', 'with', 'by', 'from', 'as', 'is', 'was', 'are', 'were', 'been', 'be', 'have', 'has', 'had', 'do', 'does', 'did', 'will', 'would', 'could', 'should', 'may', 'might', 'must', 'can'}
        filtered_words = [w for w in words if w not in stop_words and len(w) > 3]
        all_words.extend(filtered_words)
    
    word_counts = Counter(all_words).most_common(15)
    if word_counts:
        words_list = [wc[0] for wc in word_counts]
        counts_list = [wc[1] for wc in word_counts]
        
        fig3 = go.Figure(data=[
            go.Bar(
                x=words_list,
                y=counts_list,
                marker_color='rgb(255, 127, 14)'
            )
        ])
        fig3.update_layout(
            title='Most Common Words in Task Goals',
            xaxis_title='Word',
            yaxis_title='Frequency',
            height=400
        )
        figures.append(fig3)
    
    return figures, None


print("✅ Data Dashboard functions ready.")

print("📚 RAG Handlers ready.")

In [ ]:
# ============================================================
# CELL 7 — 16-LAYER DEEP PIPELINE (L0-L10)
# ============================================================
# The 4CBON Runtime Engine - Layered Cognitive Execution System
# Pipeline: L0 → P → W → LX → LA → LC → L1 → L2 → LP → L3 → L4 → LR → L6 → L7 → L8 → L9 → L10
# ============================================================

import uuid
from typing import Generator, Dict, Any, List, Optional

# Layer definitions
PIPELINE_LAYERS = [
    {"id": "L0", "name": "Interpretation Engine", "color": "#ff6b35", "emoji": "◎"},
    {"id": "P", "name": "Parsing Layer", "color": "#a855f7", "emoji": "⊞"},
    {"id": "W", "name": "World Model Layer", "color": "#00d4ff", "emoji": "⊕"},
    {"id": "LX", "name": "Reality Adjudication", "color": "#f97316", "emoji": "⊛"},
    {"id": "LA", "name": "Adversarial Countermodel", "color": "#dc2626", "emoji": "⚔"},
    {"id": "LC", "name": "Compression Integrity", "color": "#0ea5e9", "emoji": "⊘"},
    {"id": "L1", "name": "Hypothesis Engine", "color": "#38bdf8", "emoji": "◈"},
    {"id": "L2", "name": "Evaluation Layer", "color": "#f59e0b", "emoji": "◉"},
    {"id": "LP", "name": "Policy Translation", "color": "#8b5cf6", "emoji": "⊛"},
    {"id": "L3", "name": "Rewrite Planner", "color": "#7c3aed", "emoji": "◐"},
    {"id": "L4", "name": "Finalization Engine", "color": "#10b981", "emoji": "★", "final": True},
    {"id": "LR", "name": "Regret Layer", "color": "#ef4444", "emoji": "◑"},
    {"id": "L6", "name": "Trace Memory", "color": "#f43f5e", "emoji": "⟳"},
    {"id": "L7", "name": "Curriculum Generator", "color": "#c084fc", "emoji": "◆"},
    {"id": "L8", "name": "Identity Model", "color": "#fbbf24", "emoji": "⚙"},
    {"id": "L9", "name": "Socratic Integrity", "color": "#38bdf8", "emoji": "?"},
    {"id": "L10", "name": "Synthesis/Audit", "color": "#6ee7b7", "emoji": "✦"},
]

# Runtime specification
RUNTIME_SPEC = """You are the 4CBON Runtime Engine — a layered cognitive execution system.

Your job is to process AI-generated answers through a deterministic multi-layer transformation pipeline. You execute one layer at a time. Each layer has a specific cognitive role. You never skip layers. You never merge layers.

PIPELINE: L0 → P → W → LX → LA → LC → L1 → L2 → L3 → L4 → LR → L6 → L7 → L8 → L9 → L10

YOUR IDENTITY:
- You are not a chatbot. You are an execution engine.
- Every output is a cognitive artifact, not a conversation.
- You think in transformations, not responses.
- You are transparent. Every reasoning step is visible.
- You improve answers systematically, not randomly.

LAYER DEFINITIONS:
L0 — INTERPRETATION ENGINE: Understand the input. Infer intent. Extract task type, constraints, ambiguities. Define what excellent looks like.
P  — PARSING LAYER: Break the input into logical units. Identify claims, structure, gaps, missing logic.
W  — WORLD MODEL LAYER: Extract factual claims. Separate certainty: high / medium / unknown. Integrate validated external critiques as HIGH certainty facts.
LX — REALITY ADJUDICATION LAYER: For every claim flagged MEDIUM or UNKNOWN by W, ask: (1) What prediction would this claim make that could be tested? (2) What would an adversary say against it? (3) What external artifact would verify or falsify it? Label each claim: FALSIFIABLE / UNFALSIFIABLE / TESTABLE-IN-PRINCIPLE. Claims that cannot answer any question get labeled UNGROUNDED.
LA — ADVERSARIAL COUNTERMODEL LAYER: Actively attempt to structurally destroy the answer's core claims. Generate: (1) the strongest competing explanation, (2) hidden assumptions, (3) conditions under which the answer is completely wrong, (4) simplest alternative.
LC — COMPRESSION INTEGRITY LAYER: Hunt semantic smoothing. Detect where: (1) multiple concepts collapsed into one term, (2) metaphor replaced mechanism, (3) elegance erased uncertainty, (4) abstraction hid causality.
L1 — HYPOTHESIS ENGINE: Generate 2-3 interpretations of how this answer could be improved. Include a failure mode hypothesis.
L2 — EVALUATION LAYER: Score the hypotheses. Identify contradictions, gaps. Pick the best path forward.
L3 — REWRITE PLANNER: Plan the rewrite. Decide what stays, changes, gets added.
L4 — FINALIZATION ENGINE: Execute the rewrite. Produce the final improved answer.
LR — REGRET LAYER: Analyze improvement delta. What errors corrected? What hallucinations removed? What still needs work?
L6 — TRACE MEMORY: Store the immutable execution log.
L7 — CURRICULUM GENERATOR: Extract lessons learned, failure patterns, reusable heuristics.
L8 — IDENTITY MODEL: Summarize system behavior this run. Strengths, weaknesses, bias tendencies.
L9 — SOCRATIC INTEGRITY ENGINE: Generate exactly 3 self-questions specific to this run.
L10 — SYNTHESIS/AUDIT LAYER: Read all prior layer outputs. Produce a final certification.

Stay in your assigned layer. Output only what that layer produces. Be precise and concise."""

class PipelinePrompts:
    @staticmethod
    def L0(answer: str, ctx: str = "") -> str:
        return f"{f'Context/Goal: {ctx}\n\n' if ctx else ''}AI ANSWER:\n{answer}\n\nYou are L0 — Interpretation Engine. Identify: task type, intent, constraints, ambiguities. Define what an excellent version of this answer looks like."
    
    @staticmethod
    def P(answer: str, l0: str) -> str:
        return f"AI ANSWER:\n{answer}\n\nL0 Interpretation:\n{l0}\n\nYou are P — Parsing Layer. Break the answer into logical units. List: (1) claims made, (2) structure used, (3) what is missing, (4) what is weak."
    
    @staticmethod
    def W(answer: str) -> str:
        return f"AI ANSWER:\n{answer}\n\nYou are W — World Model Layer. Extract the factual claims in this answer. For each claim, label certainty: HIGH / MEDIUM / UNKNOWN."
    
    @staticmethod
    def LX(answer: str, w: str) -> str:
        return f"AI ANSWER:\n{answer}\n\nW WORLD MODEL:\n{w}\n\nYou are LX — Reality Adjudication Layer. For every claim labeled MEDIUM or UNKNOWN, apply three tests: (1) PREDICTION TEST, (2) ADVERSARY TEST, (3) VERIFICATION TEST. Label: FALSIFIABLE / UNFALSIFIABLE / TESTABLE-IN-PRINCIPLE / UNGROUNDED."
    
    @staticmethod
    def LA(answer: str, lx: str) -> str:
        return f"AI ANSWER:\n{answer}\n\nREALITY AUDIT:\n{lx}\n\nYou are LA — Adversarial Countermodel Layer. Generate: (1) strongest competing explanation, (2) hidden assumptions, (3) collapse conditions, (4) simplicity challenge, (5) the collapse question."
    
    @staticmethod
    def LC(answer: str, la: str) -> str:
        return f"AI ANSWER:\n{answer}\n\nADVERSARIAL FINDINGS:\n{la}\n\nYou are LC — Compression Integrity Layer. Hunt for: (1) concept collapse, (2) metaphor substitution, (3) elegance erasure, (4) abstraction hiding causality."
    
    @staticmethod
    def L1(answer: str, p: str, w: str, lx: str, la: str, lc: str) -> str:
        return f"AI ANSWER:\n{answer}\n\nParsing:\n{p}\n\nWorld Model:\n{w}\n\nReality Audit (LX):\n{lx}\n\nAdversarial Findings (LA):\n{la}\n\nCompression Audit (LC):\n{lc}\n\nYou are L1 — Hypothesis Engine. Generate exactly 3 improvement hypotheses:
H1: [strongest improvement path]
H2: [radical reframe]
H3: [failure mode hypothesis]"
    
    @staticmethod
    def L2(l1: str, s0: int) -> str:
        return f"Hypotheses:\n{l1}\n\nInput score: {s0}/100\n\nYou are L2 — Evaluation Layer. Score each hypothesis. Identify contradictions. Pick the best path forward."
    
    @staticmethod
    def LP(answer: str, l2: str) -> str:
        return f'Claim: "{answer[:200]}"\nProposal: "{l2[:200]}"\n\nDoes Proposal say the OPPOSITE of Claim? Answer: YES or NO'
    
    @staticmethod
    def L3(answer: str, l2: str, w: str) -> str:
        return f"Best path:\n{l2}\n\nWorld facts:\n{w}\n\nOriginal answer:\n{answer}\n\nYou are L3 — Rewrite Planner. Create a precise rewrite brief: (1) what stays, (2) what changes, (3) what gets added, (4) what gets removed."
    
    @staticmethod
    def L4(answer: str, l3: str, w: str) -> str:
        return f"ORIGINAL ANSWER:\n{answer}\n\nREWRITE PLAN:\n{l3}\n\nWORLD FACTS:\n{w}\n\nYou are L4 — Finalization Engine. Execute the rewrite plan. Produce the final improved answer."
    
    @staticmethod
    def LR(answer: str, l4: str, s0: int, s1: int) -> str:
        return f"BEFORE (score {s0}/100):\n{answer}\n\nAFTER (score {s1}/100):\n{l4}\n\nYou are LR — Regret Layer. Analyze: (1) errors corrected, (2) hallucinations removed, (3) structural improvements, (4) what still needs work."
    
    @staticmethod
    def L6(s0: int, s1: int, gaps: list) -> str:
        return f"Score trajectory: {s0} → {s1}\nGaps fixed: {', '.join(gaps) if gaps else 'none'}\n\nYou are L6 — Trace Memory. Write the immutable execution log of this run."
    
    @staticmethod
    def L7(lr: str, l6: str) -> str:
        return f"Regret analysis:\n{lr}\n\nTrace:\n{l6}\n\nYou are L7 — Curriculum Generator. Extract: (1) 3 lessons learned, (2) key failure patterns, (3) 2 reusable heuristics, (4) 2 challenge questions."
    
    @staticmethod
    def L8(s0: int, s1: int, gaps: list) -> str:
        return f"Run: score {s0}→{s1}, gaps fixed: {', '.join(gaps) if gaps else 'none'}\n\nYou are L8 — Identity Model. Summarize: 1. Strengths, 2. Weaknesses, 3. Bias tendencies, 4. One new self-belief"
    
    @staticmethod
    def L9(l8: str, s0: int, s1: int, l4: str) -> str:
        return f"You just completed a pipeline run. Score: {s0}→{s1}.\n\nL8 self-belief:\n{l8[:400]}\n\nL4 final rewrite:\n{l4[:300]}\n\nYou are L9 — Socratic Integrity Engine. Generate exactly 3 questions: Q: [observational], Q: [reasoning], Q: [alignment]"
    
    @staticmethod
    def L10(l4: str, lr: str, l7: str, l8: str, l9qs: str, s0: int, s1: int) -> str:
        return f"""PIPELINE RUN SUMMARY:
Score: {s0} → {s1}

L4 FINAL REWRITE:
{l4}

LR REGRET ANALYSIS:
{lr[:400]}

L7 LESSONS:
{l7[:300]}

L8 SELF-BELIEF:
{l8[:200]}

L9 UNRESOLVED QUESTIONS:
{l9qs}

You are L10 — Synthesis/Audit Layer. Produce a final certification addressing:
1. IMPROVEMENT VERDICT: Did the rewrite genuinely improve the answer?
2. CONTRADICTION AUDIT: Did any layer contradict another?
3. INTEGRITY CHECK: Does L4 contain remaining overclaims or hallucinations?
4. HUMAN VERDICT: One sentence a human should read. Start with CERTIFIED, CERTIFIED WITH CAUTION, or REQUIRES REVIEW."""


def score_text(text: str, original_score: int = None) -> int:
    """Score text quality using Gemini."""
    global GEMINI_API_KEY
    
    if original_score is not None:
        prompt = f"You are judging a REWRITE of an AI answer. The original scored {original_score}/100.
Rate only whether this rewrite improved the original.
Return a single integer 0-100 where 50 = no change, above 50 = better, below 50 = worse.
ANSWER:
{text[:1200]}
Reply with ONLY a single integer 0-100."
    else:
        prompt = f"Rate the quality of this AI-generated answer 0-100.
Criteria: Clarity (0-25), Structure (0-25), Depth (0-25), Correctness (0-25).
ANSWER:
{text[:1200]}
Reply with ONLY a single integer 0-100."
    
    try:
        result = call_gemini("You are a quality scorer. Reply with only numbers.", prompt, max_tokens=10)
        num = int(''.join(c for c in result if c.isdigit()))
        return min(100, max(0, num)) if num else 50
    except:
        return 50

def execute_deep_pipeline(answer: str, context: str = "") -> Generator[Dict[str, Any], None, None]:
    """
    Execute the 16-layer deep pipeline.
    Yields events as they happen for streaming to the UI.
    """
    global GEMINI_API_KEY
    
    # Check if we have API access
    if not GEMINI_API_KEY and RUNTIME_MODE == "VERCEL":
        yield {"type": "error", "message": "Gemini API key required for Deep Pipeline. Call set_gemini_key() first."}
        return
    
    run_id = f"run_{int(time.time())}_{uuid.uuid4().hex[:8]}"
    outputs = {}
    
    yield {"type": "start", "run_id": run_id, "mode": "DEEP_PIPELINE"}
    
    # Score original
    yield {"type": "layer_start", "layer": "score_before"}
    s0 = score_text(answer)
    yield {"type": "score_before", "score": s0}
    
    operating_mode = "HIGH_QUALITY" if s0 >= 68 else "STANDARD"
    yield {"type": "mode", "mode": operating_mode}
    
    # Layer execution
    layers_to_run = [
        ("L0", PipelinePrompts.L0(answer, context), 800),
        ("P", PipelinePrompts.P(answer, ""), 800),
        ("W", PipelinePrompts.W(answer), 800),
        ("LX", PipelinePrompts.LX(answer, ""), 800),
        ("LA", PipelinePrompts.LA(answer, ""), 800),
        ("LC", PipelinePrompts.LC(answer, ""), 800),
    ]
    
    # Run first 6 layers
    for layer_id, prompt, max_tokens in layers_to_run:
        yield {"type": "layer_start", "layer": layer_id}
        system = f"{RUNTIME_SPEC}\n\nYOU ARE NOW EXECUTING: {layer_id}\nStay in this layer only."
        result = call_gemini(system, prompt, max_tokens=max_tokens)
        outputs[layer_id] = result
        yield {"type": "layer_complete", "layer": layer_id, "output": result}
    
    # L1 - Hypothesis Engine with all context
    yield {"type": "layer_start", "layer": "L1"}
    l1_prompt = PipelinePrompts.L1(
        answer,
        outputs.get("P", ""),
        outputs.get("W", ""),
        outputs.get("LX", "")[:600],
        outputs.get("LA", "")[:600],
        outputs.get("LC", "")[:600]
    )
    l1 = call_gemini(f"{RUNTIME_SPEC}\n\nYOU ARE NOW EXECUTING: L1", l1_prompt, max_tokens=800)
    outputs["L1"] = l1
    yield {"type": "layer_complete", "layer": "L1", "output": l1}
    
    # L2 - Evaluation Layer
    yield {"type": "layer_start", "layer": "L2"}
    l2_prompt = PipelinePrompts.L2(l1, s0)
    l2_max_tokens = 50 if operating_mode == "HIGH_QUALITY" else 800
    l2 = call_gemini(f"{RUNTIME_SPEC}\n\nYOU ARE NOW EXECUTING: L2", l2_prompt, max_tokens=l2_max_tokens)
    outputs["L2"] = l2
    yield {"type": "layer_complete", "layer": "L2", "output": l2}
    
    # L2 halt checks
    if "NO_REWRITE" in l2 or "PRESERVE" in l2 or "ESCALATE" in l2:
        yield {"type": "score_after", "score": s0}
        reason = "HIGH QUALITY MODE: No improvement found." if "NO_REWRITE" in l2 else "L2 PRESERVE/ESCALATE"
        yield {"type": "halt", "reason": reason}
        yield {"type": "complete", "run_id": run_id, "score_before": s0, "score_after": s0}
        return
    
    # LP - Policy Translation
    yield {"type": "layer_start", "layer": "LP"}
    lp = call_gemini(f"{RUNTIME_SPEC}\n\nYOU ARE NOW EXECUTING: LP", PipelinePrompts.LP(answer, l2), max_tokens=5)
    outputs["LP"] = lp
    yield {"type": "layer_complete", "layer": "LP", "output": lp}
    
    if lp.strip().upper().startswith("YES"):
        yield {"type": "score_after", "score": s0}
        yield {"type": "halt", "reason": "LP HALT: proposed change inverts the original claim."}
        yield {"type": "complete", "run_id": run_id, "score_before": s0, "score_after": s0}
        return
    
    # L3 - Rewrite Planner
    yield {"type": "layer_start", "layer": "L3"}
    l3 = call_gemini(f"{RUNTIME_SPEC}\n\nYOU ARE NOW EXECUTING: L3", PipelinePrompts.L3(answer, l2, outputs.get("W", "")), max_tokens=800)
    outputs["L3"] = l3
    yield {"type": "layer_complete", "layer": "L3", "output": l3}
    
    # L4 - Finalization Engine
    yield {"type": "layer_start", "layer": "L4"}
    l4 = call_gemini(f"{RUNTIME_SPEC}\n\nYOU ARE NOW EXECUTING: L4", PipelinePrompts.L4(answer, l3, outputs.get("W", "")), max_tokens=2500)
    outputs["L4"] = l4
    yield {"type": "layer_complete", "layer": "L4", "output": l4}
    
    # L4 failure check
    if not l4 or len(l4.strip()) < 50:
        yield {"type": "score_after", "score": s0}
        yield {"type": "halt", "reason": "L4 HALT: Execution failed."}
        yield {"type": "complete", "run_id": run_id, "score_before": s0, "score_after": s0}
        return
    
    # Score rewrite
    yield {"type": "scoring"}
    s1 = score_text(l4, original_score=s0)
    yield {"type": "score_after", "score": s1}
    
    gaps_fixed = ["clarity", "structure", "depth"] if s1 > s0 else []
    
    # Remaining layers
    remaining_layers = [
        ("LR", PipelinePrompts.LR(answer, l4, s0, s1), 800),
        ("L6", PipelinePrompts.L6(s0, s1, gaps_fixed), 800),
        ("L7", PipelinePrompts.L7("", ""), 2500),
        ("L8", PipelinePrompts.L8(s0, s1, gaps_fixed), 800),
    ]
    
    for layer_id, prompt, max_tokens in remaining_layers:
        yield {"type": "layer_start", "layer": layer_id}
        result = call_gemini(f"{RUNTIME_SPEC}\n\nYOU ARE NOW EXECUTING: {layer_id}", prompt, max_tokens=max_tokens)
        outputs[layer_id] = result
        yield {"type": "layer_complete", "layer": layer_id, "output": result}
    
    # L9 - Socratic Integrity
    yield {"type": "layer_start", "layer": "L9"}
    l9_raw = call_gemini(f"{RUNTIME_SPEC}\n\nYOU ARE NOW EXECUTING: L9", PipelinePrompts.L9(outputs.get("L8", ""), s0, s1, l4), max_tokens=300)
    l9_questions = [line.replace("Q:", "").strip() for line in l9_raw.split("\n") if line.strip().startswith("Q:")]
    l9_questions = l9_questions[:3]
    outputs["L9"] = l9_raw
    yield {"type": "layer_complete", "layer": "L9", "output": l9_raw}
    
    # L10 - Synthesis/Audit
    yield {"type": "layer_start", "layer": "L10"}
    l10 = call_gemini(
        f"{RUNTIME_SPEC}\n\nYOU ARE NOW EXECUTING: L10",
        PipelinePrompts.L10(
            l4, outputs.get("LR", ""), outputs.get("L7", ""),
            outputs.get("L8", ""), "\n".join(l9_questions) if l9_questions else "No questions",
            s0, s1
        ),
        max_tokens=800
    )
    outputs["L10"] = l10
    yield {"type": "layer_complete", "layer": "L10", "output": l10}
    
    yield {"type": "complete", "run_id": run_id, "score_before": s0, "score_after": s1}


def run_deep_pipeline(answer: str, context: str = "") -> Dict[str, Any]:
    """Run the deep pipeline and collect all outputs."""
    results = {
        "score_before": None,
        "score_after": None,
        "outputs": {},
        "run_id": None,
        "status": "pending"
    }
    
    for event in execute_deep_pipeline(answer, context):
        if event["type"] == "score_before":
            results["score_before"] = event["score"]
        elif event["type"] == "score_after":
            results["score_after"] = event["score"]
        elif event["type"] == "layer_complete":
            results["outputs"][event["layer"]] = event["output"]
        elif event["type"] == "start":
            results["run_id"] = event["run_id"]
        elif event["type"] == "complete":
            results["status"] = "complete"
        elif event["type"] == "error":
            results["status"] = f"error: {event['message']}"
        elif event["type"] == "halt":
            results["status"] = f"halted: {event['reason']}"
    
    return results

print(f"✅ Deep Pipeline ready with {len(PIPELINE_LAYERS)} layers")
print("Layers:", [l['id'] for l in PIPELINE_LAYERS])

In [ ]:
# ============================================================
# CELL 8 — Master UI (Unified Edition - All Features + Deep Pipeline)
# ============================================================
import gradio as gr
import traceback
import ast
import shutil
from pathlib import Path

try:
    gr.close_all()
except:
    pass


# Safety fallback for _parse_agent_json
try:
    _parse_agent_json
except NameError:
    import json as _json
    def _extract_balanced_fb(text, open_ch, close_ch):
        if not text:
            return None
        start = text.find(open_ch)
        if start == -1:
            return None
        depth = 0
        in_string = False
        escape = False
        for i in range(start, len(text)):
            ch = text[i]
            if in_string:
                if escape:
                    escape = False
                elif ch == '\\':
                    escape = True
                elif ch == '"':
                    in_string = False
            else:
                if ch == '"':
                    in_string = True
                elif ch == open_ch:
                    depth += 1
                elif ch == close_ch:
                    depth -= 1
                    if depth == 0:
                        return text[start:i + 1]
        return None

    def _parse_agent_json(raw_response):
        if not raw_response:
            return None
        candidate = _extract_balanced_fb(raw_response, '{', '}')
        try:
            if candidate:
                return _json.loads(candidate)
            return _json.loads(raw_response)
        except Exception:
            return None
    print("ℹ️ _parse_agent_json defined locally (safety fallback).")


# ============================================================
# BUILDER TAB FUNCTIONS
# ============================================================

def read_notebook_cells(notebook_path):
    """Read all code cells from the notebook."""
    try:
        with open(notebook_path, 'r') as f:
            nb = json.load(f)
        cells = []
        for i, cell in enumerate(nb.get('cells', [])):
            if cell.get('cell_type') == 'code':
                source = ''.join(cell.get('source', []))
                cells.append({
                    'index': i,
                    'source': source,
                    'length': len(source)
                })
        return cells, None
    except Exception as e:
        return None, str(e)


def generate_builder_proposal(notebook_path, direction):
    """Generate a proposal for notebook modifications."""
    if not direction or not direction.strip():
        return "❌ Please enter a direction.", "", "❌ No direction"
    
    cells, error = read_notebook_cells(notebook_path)
    if error:
        return f"❌ Failed to read notebook: {error}", "", "❌ Read error"
    
    notebook_context = ""
    for cell in cells:
        notebook_context += f"\n--- CELL {cell['index']} ({cell['length']} chars) ---\n"
        notebook_context += f"```python\n{cell['source']}\n```\n"
    
    prompt = f"""You are the 4CBON2 Builder. Analyze the entire notebook and propose changes.

USER DIRECTION:
{direction}

Generate a JSON proposal with:
{{
    "proposal_id": "prop_YYYYMMDD_HHMMSS",
    "direction": "...",
    "summary": "...",
    "changes": [
        {{
            "cell_index": 0,
            "section": "...",
            "action": "modify|add|replace",
            "original_code": "...",
            "new_code": "...",
            "rationale": "..."
        }}
    ]
    "instructions": "..."
}}

JSON:"""
    
    try:
        result = safe_ask_raw(prompt, max_tokens=4096)
        
        if not result or result.startswith("⚠️") or result.startswith('{"error"'):
            return f"❌ Proposal generation failed: {result}", "", "❌ Failed"
        
        parsed = _parse_agent_json(result)
        if parsed is None:
            return f"❌ Failed to parse proposal JSON.\n\nRaw: {result[:1000]}...", result, "❌ JSON Error"
        
        if "proposal_id" not in parsed:
            parsed["proposal_id"] = f"prop_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
        
        summary_md = f"## 📋 Proposal: {parsed.get('proposal_id', 'N/A')}\n\n"
        summary_md += f"**Direction:** {parsed.get('direction', 'N/A')}\n\n"
        summary_md += f"**Summary:** {parsed.get('summary', 'No summary')}\n\n"
        summary_md += f"### Changes ({len(parsed.get('changes', []))})\n\n"
        
        for i, ch in enumerate(parsed.get('changes', []), 1):
            summary_md += f"**Change {i}:**\n"
            summary_md += f"- **Cell:** {ch.get('cell_index', 'N/A')}\n"
            summary_md += f"- **Section:** {ch.get('section', 'Unknown')}\n"
            summary_md += f"- **Action:** {ch.get('action', 'modify')}\n"
            summary_md += f"- **Rationale:** {ch.get('rationale', 'Not specified')}\n\n"
        
        summary_md += f"\n### Instructions\n{parsed.get('instructions', 'No instructions')}"
        
        status = f"✅ {len(parsed.get('changes', []))} change(s) proposed."
        return summary_md, json.dumps(parsed, indent=2), status
    
    except Exception as e:
        tb = traceback.format_exc()
        return f"❌ Unexpected error: {str(e)}\n\n{tb}", "", "❌ Failed"


def validate_syntax(code):
    """Validate Python syntax."""
    try:
        ast.parse(code)
        return True, None
    except SyntaxError as e:
        return False, f"Line {e.lineno}: {e.msg}"


def five_lens_verdict(change):
    """Run 5-lens automated verdict on a proposed change."""
    prompt = f"""Evaluate this proposed code change using 5 lenses:

CELL INDEX: {change.get('cell_index', 'N/A')}
ACTION: {change.get('action', 'modify')}
RATIONALE: {change.get('rationale', 'N/A')}

NEW CODE:
```python
{change.get('new_code', '')[:2000]}
```

Evaluate: CORRECTNESS, SAFETY, COMPLETENESS, COMPATIBILITY, CLARITY.

Respond with ONLY this JSON:
{{
    "verdict": "APPROVE" or "REJECT",
    "confidence": 0.0-1.0,
    "reasoning": "Brief explanation"
}}

JSON:"""
    
    try:
        result = safe_ask_raw(prompt, max_tokens=512)
        parsed = _parse_agent_json(result)
        if parsed and "verdict" in parsed:
            return parsed
        return {"verdict": "REJECT", "confidence": 0.0, "reasoning": "Failed to parse verdict"}
    except Exception as e:
        return {"verdict": "REJECT", "confidence": 0.0, "reasoning": f"Error: {str(e)}"}


def apply_proposals(proposals_json, notebook_path):
    """Review and apply approved changes."""
    log = []
    
    try:
        proposals = json.loads(proposals_json)
    except Exception as e:
        return f"❌ Failed to parse proposals JSON: {str(e)}"
    
    changes = proposals.get('changes', [])
    if not changes:
        return "❌ No changes to apply."
    
    # Create backup
    backup_dir = Path("/content/drive/MyDrive/4cbon_notebook_backups")
    backup_dir.mkdir(parents=True, exist_ok=True)
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    backup_path = backup_dir / f"4CBOn2_Unified_backup_{timestamp}.ipynb"
    
    try:
        shutil.copy(notebook_path, backup_path)
        log.append(f"✅ Backup created: {backup_path}")
    except Exception as e:
        log.append(f"⚠️ Backup failed: {str(e)}")
    
    # Read notebook
    try:
        with open(notebook_path, 'r') as f:
            nb = json.load(f)
    except Exception as e:
        return f"❌ Failed to read notebook: {str(e)}\n\n" + "\n".join(log)
    
    # Process each change
    applied = 0
    skipped = 0
    
    for i, change in enumerate(changes, 1):
        cell_idx = change.get('cell_index')
        new_code = change.get('new_code', '')
        
        log.append(f"\n--- Change {i}: Cell {cell_idx} ---")
        
        # Validate syntax
        syntax_ok, syntax_error = validate_syntax(new_code)
        if not syntax_ok:
            log.append(f"❌ REJECTED - Syntax error: {syntax_error}")
            skipped += 1
            continue
        
        log.append("✅ Syntax validation passed")
        
        # Run 5-lens verdict
        log.append("🔍 Running 5-lens verdict...")
        verdict = five_lens_verdict(change)
        
        if verdict.get('verdict') == 'APPROVE':
            log.append(f"✅ APPROVED (confidence: {verdict.get('confidence', 0):.2f})")
            log.append(f"   Reasoning: {verdict.get('reasoning', 'N/A')}")
            applied += 1
        else:
            log.append(f"❌ REJECTED (confidence: {verdict.get('confidence', 0):.2f})")
            log.append(f"   Reasoning: {verdict.get('reasoning', 'N/A')}")
            skipped += 1
    
    log.append(f"\n{'='*50}")
    log.append(f"SUMMARY: {applied} applied, {skipped} skipped")
    log.append(f"{'='*50}")
    
    return "\n".join(log)


# ============================================================
# DEEP PIPELINE UI FUNCTIONS
# ============================================================

def run_pipeline_stream(answer, context):
    """Run the deep pipeline and yield updates."""
    if not answer or not answer.strip():
        yield {"status": "❌ Please paste an AI answer."}
        return
    
    for event in execute_deep_pipeline(answer, context):
        yield event

def format_pipeline_output(events):
    """Format pipeline events into readable output."""
    output_parts = []
    score_before = None
    score_after = None
    
    for event in events:
        if event["type"] == "start":
            output_parts.append(f"🚀 Pipeline started: {event['run_id']}")
        elif event["type"] == "mode":
            output_parts.append(f"📊 Operating mode: {event['mode']}")
        elif event["type"] == "score_before":
            score_before = event["score"]
            output_parts.append(f"\n📈 Score before: {score_before}/100")
        elif event["type"] == "layer_start":
            output_parts.append(f"\n⟳ Running {event['layer']}...")
        elif event["type"] == "layer_complete":
            layer = event["layer"]
            output = event["output"][:300] + "..." if len(event["output"]) > 300 else event["output"]
            output_parts.append(f"✅ {layer}: {output}")
        elif event["type"] == "score_after":
            score_after = event["score"]
        elif event["type"] == "scoring":
            output_parts.append("\n⏳ Scoring rewrite...")
        elif event["type"] == "halt":
            output_parts.append(f"\n⚠️ HALTED: {event['reason']}")
        elif event["type"] == "error":
            output_parts.append(f"\n❌ ERROR: {event['message']}")
        elif event["type"] == "complete":
            delta = score_after - score_before if score_before and score_after else 0
            output_parts.append(f"\n✅ COMPLETE! Score: {score_before}/100 → {score_after}/100 ({'+' if delta > 0 else ''}{delta})")
    
    return "\n".join(output_parts)


# ============================================================
# UNIFIED GRADIO INTERFACE
# ============================================================

with gr.Blocks(title="4CBON2 — Unified Cognitive Platform") as demo:
    gr.Markdown("# 🚀 4CBON2 — Unified Cognitive Platform")
    gr.Markdown(f"*Mode: {RUNTIME_MODE} | Model: {MODEL_NAME} | Features: Agents + RAG + Builder + Deep Pipeline*")

    with gr.Tabs():
        # ═══ Upload Tab ═══
        with gr.TabItem("📁 Upload Documents"):
            gr.Markdown("Upload .txt, .pdf, or .docx files to the knowledge base.")
            file_input = gr.File(label="Upload file", file_types=[".txt", ".pdf", ".docx"])
            upload_output = gr.Textbox(label="Status", interactive=False)
            upload_btn = gr.Button("Process & Index", variant="primary")
            upload_btn.click(fn=process_document, inputs=[file_input], outputs=[upload_output])

        # ═══ Ask a Question Tab ═══
        with gr.TabItem("❓ Ask a Question"):
            gr.Markdown("Ask a question. The system searches the knowledge base and answers with the 5-lens framework.")
            question_box = gr.Textbox(label="Your Question", lines=3, placeholder="What is the hard problem of consciousness?")
            ask_output = gr.Textbox(label="Answer", lines=20, interactive=False)
            ask_status = gr.Textbox(label="Status", interactive=False)
            ask_btn = gr.Button("Ask", variant="primary")

            def ask_five_lens(question):
                if not question or not question.strip():
                    return "❌ Enter a question.", "❌ No question"
                try:
                    answer = handle_ask_question(COLLECTION_NAME, question)
                    status = "✅ Done" if not answer.startswith("❌") else answer
                    return answer, status
                except Exception as e:
                    return f"❌ Error: {str(e)}", "❌ Failed"

            ask_btn.click(fn=ask_five_lens, inputs=[question_box], outputs=[ask_output, ask_status])

        # ═══ Agent Mode Tab ═══
        with gr.TabItem("🤖 Agent Mode"):
            gr.Markdown("Multi-Agent Orchestration with 12 specialists.")
            with gr.Row():
                with gr.Column(scale=2):
                    profile_selector = gr.Dropdown(choices=list(AGENT_PROFILES.keys()), value="New Autonomous Agent", label="Agent Profile")
                    agent_goal = gr.Textbox(label="Goal / Instructions", lines=3, placeholder="e.g. Analyze our competitor positioning and recommend a content strategy...")
                    agent_btn = gr.Button("Run Orchestrator", variant="primary")
                
                with gr.Column(scale=1):
                    gr.Markdown("### API Configuration")
                    api_key_input = gr.Textbox(label="Gemini API Key (optional)", type="password", placeholder="Required only for Vercel mode")
                    mode_display = gr.Markdown(f"**Current Mode:** {RUNTIME_MODE}")
            
            agent_output = gr.Textbox(label="Execution Log & Output", lines=25, interactive=False)
            
            def run_agent_with_key(goal, api_key):
                if api_key and api_key.strip():
                    set_gemini_key(api_key)
                yield ""
                for chunk in run_orchestrator_stream(goal):
                    yield chunk
            
            agent_btn.click(
                fn=run_agent_with_key,
                inputs=[agent_goal, api_key_input],
                outputs=agent_output
            )

        # ═══ Deep Pipeline Tab (NEW!) ═══
        with gr.TabItem("🧠 Deep Pipeline (L0-L10)"):
            gr.Markdown("""
            ## 🧠 16-Layer Cognitive Pipeline
            
            Paste any AI-generated answer. The pipeline runs it through **16 cognitive layers** to measurably improve it.
            
            **Pipeline:** L0 → P → W → LX → LA → LC → L1 → L2 → LP → L3 → L4 → LR → L6 → L7 → L8 → L9 → L10
            
            **Features:**
            - Interpretation, Parsing, World Model, Reality Adjudication
            - Adversarial Attack, Compression Integrity
            - Hypothesis Generation, Evaluation, Rewrite Planning
            - Finalization, Regret Analysis, Memory
            - Learning, Identity, Socratic Questions, Final Audit
            """)
            
            with gr.Row():
                with gr.Column(scale=2):
                    pipeline_answer = gr.Textbox(
                        label="Paste AI Answer",
                        lines=6,
                        placeholder="Paste any AI-generated answer here..."
                    )
                    pipeline_context = gr.Textbox(
                        label="Context (optional)",
                        lines=2,
                        placeholder="What should this answer achieve?"
                    )
                    pipeline_api_key = gr.Textbox(
                        label="Gemini API Key",
                        type="password",
                        placeholder="Required for Deep Pipeline"
                    )
                    pipeline_btn = gr.Button("▶ RUN PIPELINE", variant="primary")
                
                with gr.Column(scale=1):
                    pipeline_status = gr.Markdown("**Status:** Ready")
                    gr.Markdown("""
                    **Layer Progress:**
                    - L0: Interpretation
                    - P: Parsing
                    - W: World Model
                    - LX: Reality Adjudication
                    - LA: Adversarial
                    - LC: Compression
                    - L1-L2: Hypothesis/Evaluation
                    - LP: Policy Check
                    - L3-L4: Rewrite/Finalize
                    - LR-L10: Memory & Audit
                    """)
            
            pipeline_output = gr.Textbox(label="Pipeline Output", lines=30, interactive=False)
            
            def run_pipeline_with_key(answer, context, api_key):
                if api_key and api_key.strip():
                    set_gemini_key(api_key)
                
                if not GEMINI_API_KEY:
                    yield "❌ Gemini API key required. Enter your key above or run in Colab Mode."
                    return
                
                yield "🚀 Starting pipeline...\n"
                for event in execute_deep_pipeline(answer, context):
                    if event["type"] == "layer_start":
                        yield f"⟳ Running {event['layer']}...\n"
                    elif event["type"] == "layer_complete":
                        layer = event["layer"]
                        output = event["output"][:200] + "..." if len(event["output"]) > 200 else event["output"]
                        yield f"✅ {layer}: {output}\n\n"
                    elif event["type"] == "score_before":
                        yield f"📈 Score before: {event['score']}/100\n"
                    elif event["type"] == "score_after":
                        yield f"📉 Score after: {event['score']}/100\n"
                    elif event["type"] == "scoring":
                        yield "⏳ Scoring rewrite...\n"
                    elif event["type"] == "halt":
                        yield f"⚠️ HALTED: {event['reason']}\n"
                    elif event["type"] == "error":
                        yield f"❌ ERROR: {event['message']}\n"
                    elif event["type"] == "complete":
                        delta = event["score_after"] - event["score_before"]
                        yield f"\n✅ COMPLETE! Run ID: {event['run_id']}\n"
                        yield f"📊 Score: {event['score_before']}/100 → {event['score_after']}/100 ({'+' if delta > 0 else ''}{delta})"
            
            pipeline_btn.click(
                fn=run_pipeline_with_key,
                inputs=[pipeline_answer, pipeline_context, pipeline_api_key],
                outputs=pipeline_output
            )

        # ═══ Builder Tab ═══
        with gr.TabItem("🔧 Builder"):
            gr.Markdown("""
            ## 🔧 Automated Notebook Builder
            
            This tool reads your entire notebook, proposes changes, and applies them with safety checks.
            """)
            
            with gr.Tabs():
                with gr.TabItem("📝 Generate Proposal"):
                    builder_notebook_path = gr.Textbox(
                        label="Notebook Path",
                        value="4CBON2_Unified.ipynb",
                        placeholder="Path to the notebook file"
                    )
                    builder_direction = gr.Textbox(
                        label="Direction",
                        lines=4,
                        placeholder="Describe what you want to change or add..."
                    )
                    builder_generate_btn = gr.Button("🚀 Generate Proposal", variant="primary")
                    builder_summary = gr.Markdown(value="*Proposal summary will appear here...*")
                    builder_proposals_json = gr.Textbox(
                        label="Proposals JSON",
                        lines=15,
                        interactive=True,
                        visible=False
                    )
                    builder_status = gr.Textbox(label="Status", interactive=False)
                    
                    builder_generate_btn.click(
                        fn=generate_builder_proposal,
                        inputs=[builder_notebook_path, builder_direction],
                        outputs=[builder_summary, builder_proposals_json, builder_status]
                    )
                
                with gr.TabItem("✅ Review & Apply"):
                    gr.Markdown("Review and apply approved changes.")
                    review_proposals_json = gr.Textbox(
                        label="Proposals JSON (editable)",
                        lines=20,
                        placeholder="Paste or edit proposals JSON here..."
                    )
                    review_notebook_path = gr.Textbox(
                        label="Notebook Path",
                        value="4CBON2_Unified.ipynb"
                    )
                    review_apply_btn = gr.Button("⚡ Run Automated Review & Apply", variant="primary")
                    review_log = gr.Textbox(
                        label="Transparency Log",
                        lines=25,
                        interactive=False
                    )
                    
                    review_apply_btn.click(
                        fn=apply_proposals,
                        inputs=[review_proposals_json, review_notebook_path],
                        outputs=[review_log]
                    )

        # ═══ Data Dashboard Tab ═══
        with gr.TabItem("📊 Data Dashboard"):
            gr.Markdown("""
            ## 📊 Task Memory Visualization
            
            Visualize your agent task history with interactive Plotly charts.
            """)
            
            dashboard_btn = gr.Button("🔄 Load Dashboard", variant="primary")
            dashboard_output = gr.Textbox(label="Status", interactive=False)
            
            with gr.Row():
                dashboard_plot1 = gr.Plot(label="Task Timeline")
                dashboard_plot2 = gr.Plot(label="Subtask Distribution")
            
            with gr.Row():
                dashboard_plot3 = gr.Plot(label="Goal Word Frequency")
            
            def load_dashboard():
                try:
                    figures, error = create_plotly_dashboard()
                    if error:
                        return f"❌ {error}", None, None, None
                    
                    if not figures:
                        return "❌ No figures generated", None, None, None
                    
                    fig1 = figures[0] if len(figures) > 0 else None
                    fig2 = figures[1] if len(figures) > 1 else None
                    fig3 = figures[2] if len(figures) > 2 else None
                    
                    return f"✅ Loaded {len(figures)} visualization(s)", fig1, fig2, fig3
                except Exception as e:
                    return f"❌ Error: {str(e)}", None, None, None
            
            dashboard_btn.click(
                fn=load_dashboard,
                inputs=[],
                outputs=[dashboard_output, dashboard_plot1, dashboard_plot2, dashboard_plot3]
            )

        # ═══ Agent Status Tab ═══
        with gr.TabItem("📋 Agent Status"):
            gr.Markdown("View all agent conversation histories.")
            refresh_btn = gr.Button("Refresh")
            agent_status_display = gr.Markdown("Click refresh to load.")
            
            def get_agent_status():
                output = "## 📋 Agent Status\n\n"
                for agent_id in get_all_agents():
                    agent = load_agent(agent_id)
                    history_len = len(agent.get("conversation_history", []))
                    output += f"- **{agent_id}**: {history_len} messages\n"
                return output
            
            refresh_btn.click(fn=get_agent_status, outputs=[agent_status_display])
            demo.load(fn=get_agent_status, outputs=[agent_status_display])

demo.queue()
demo.launch(inline=False, share=True)


In [ ]:
# ============================================================
# CELL 9 — Download This Notebook
# ============================================================
from google.colab import files
files.download('4CBON2_Unified.ipynb')